In [1]:
import pandas as pd
import numpy as np
import boto3
from pyathena import connect

# Connect to Athena
conn = connect(
    s3_staging_dir="s3://mexico-public-safety-observatory/athena-results/",
    region_name="us-east-1"
)

# Read data
query = """
    SELECT anio, month, entidad_nombre, tipo_delito, rate_per_100k
    FROM observatory.delitos_rate
    WHERE rate_per_100k IS NOT NULL
"""

df = pd.read_sql(query, conn)
print(df.shape)
df.head()

/tmp/ipykernel_4624/4235394606.py:19: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


(54912, 5)


,anio,month,entidad_nombre,tipo_delito,rate_per_100k
0,2015,3,Aguascalientes,Homicidio,1.264270
1,2015,2,Aguascalientes,Lesiones,17.699786
2,2015,3,Aguascalientes,Lesiones,21.715704
3,2015,5,Aguascalientes,Lesiones,23.574926
4,2015,6,Aguascalientes,Lesiones,24.541721


Son 54,912 filas y es lo que esperábamos por los 32 estados x 13 tipos de delito x 11 años x 12 meses

In [2]:
# Pivot: aggregate by state and crime type (average rate across all months/years)
df_pivot = df.groupby(['entidad_nombre', 'tipo_delito'])['rate_per_100k'].mean().reset_index()

# Wide format: states as rows, crime types as columns
df_wide = df_pivot.pivot(index='entidad_nombre', columns='tipo_delito', values='rate_per_100k').fillna(0)

print(df_wide.shape)
df_wide.head()

(32, 13)


tipo_delito,Aborto,Corrupción de menores,Extorsión,Feminicidio,Homicidio,Lesiones,Otros delitos contra la sociedad,Otros delitos que atentan contra la libertad personal,Otros delitos que atentan contra la vida y la integridad corporal,Rapto,Secuestro,Trata de personas,Tráfico de menores
entidad_nombre,,,,,,,,,,,,,
Aguascalientes,0.047192,0.400292,0.584336,0.022979,1.534479,25.028850,0.012866,1.531092,0.267852,0.000000,0.039567,0.026924,0.003692
Baja California,0.110113,1.045459,0.349997,0.050447,6.131680,25.130552,0.100115,3.361645,4.156381,0.000222,0.037110,0.085377,0.005389
Baja California Sur,0.093934,0.515305,1.405292,0.031814,2.693981,21.751489,0.007591,1.480846,1.809190,0.001014,0.023861,0.045969,0.000000
Campeche,0.011294,0.150819,0.348838,0.057789,1.653326,12.681221,0.028148,0.967253,0.961816,0.000000,0.037417,0.025265,0.000000
Chiapas,0.015156,0.073133,0.184419,0.048339,2.042140,2.642990,0.138546,0.245121,0.175307,0.004435,0.048775,0.055001,0.002588


In [3]:
# Standarization and PCA
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Standardize: each crime type has mean 0 and std 1
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_wide)

# PCA
pca = PCA()
X_pca = pca.fit_transform(X_scaled)

# Variance explained by each component
explained = pca.explained_variance_ratio_
for i, var in enumerate(explained):
    print(f"PC{i+1}: {var:.3f} ({var*100:.1f}%)")

PC1: 0.255 (25.5%)
PC2: 0.142 (14.2%)
PC3: 0.118 (11.8%)
PC4: 0.102 (10.2%)
PC5: 0.089 (8.9%)
PC6: 0.075 (7.5%)
PC7: 0.059 (5.9%)
PC8: 0.050 (5.0%)
PC9: 0.039 (3.9%)
PC10: 0.025 (2.5%)
PC11: 0.019 (1.9%)
PC12: 0.018 (1.8%)
PC13: 0.007 (0.7%)


In [4]:
# Cumulative variance explained (~70%)
cumulative = np.cumsum(explained)
for i, cum in enumerate(cumulative):
    print(f"PC1 a PC{i+1}: {cum:.3f} ({cum*100:.1f}%)")

PC1 a PC1: 0.255 (25.5%)
PC1 a PC2: 0.398 (39.8%)
PC1 a PC3: 0.516 (51.6%)
PC1 a PC4: 0.618 (61.8%)
PC1 a PC5: 0.707 (70.7%)
PC1 a PC6: 0.782 (78.2%)
PC1 a PC7: 0.841 (84.1%)
PC1 a PC8: 0.891 (89.1%)
PC1 a PC9: 0.930 (93.0%)
PC1 a PC10: 0.956 (95.6%)
PC1 a PC11: 0.975 (97.5%)
PC1 a PC12: 0.993 (99.3%)
PC1 a PC13: 1.000 (100.0%)


In [5]:
# Use first 5 components
pca5 = PCA(n_components=5)
X_pca5 = pca5.fit_transform(X_scaled)

# Weighted score: weight each PC by its explained variance
weights = pca5.explained_variance_ratio_
raw_score = X_pca5 @ weights

# Normalize to 0-100
score_min = raw_score.min()
score_max = raw_score.max()
score_100 = (raw_score - score_min) / (score_max - score_min) * 100

# Build result dataframe
df_scores = pd.DataFrame({
    'entidad_nombre': df_wide.index,
    'violence_score': score_100
}).sort_values('violence_score', ascending=False)

print(df_scores)

                     entidad_nombre  violence_score
18                       Nuevo León      100.000000
8                            Colima       86.683358
22                     Quintana Roo       82.770812
15                          Morelos       82.541975
31                        Zacatecas       79.519716
1                   Baja California       69.298142
24                          Sinaloa       63.857385
26                          Tabasco       63.768295
5                         Chihuahua       62.909854
27                       Tamaulipas       62.382713
12                          Hidalgo       56.925610
2               Baja California Sur       53.046589
16                           México       52.790884
6                  Ciudad de México       51.688359
29  Veracruz de Ignacio de la Llave       48.777150
11                         Guerrero       44.659177
23                  San Luis Potosí       38.468643
25                           Sonora       36.068321
14          

In [6]:
from sklearn.cluster import KMeans

# K-Means with 4 clusters (low, medium, high, critical)
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df_scores['cluster'] = kmeans.fit_predict(df_scores[['violence_score']])

# Label clusters based on mean score
cluster_means = df_scores.groupby('cluster')['violence_score'].mean().sort_values()
label_map = {
    cluster_means.index[0]: 'Bajo',
    cluster_means.index[1]: 'Medio',
    cluster_means.index[2]: 'Alto',
    cluster_means.index[3]: 'Crítico'
}
df_scores['nivel'] = df_scores['cluster'].map(label_map)

print(df_scores.sort_values('violence_score', ascending=False))

                     entidad_nombre  violence_score  cluster    nivel
18                       Nuevo León      100.000000        2  Crítico
8                            Colima       86.683358        2  Crítico
22                     Quintana Roo       82.770812        2  Crítico
15                          Morelos       82.541975        2  Crítico
31                        Zacatecas       79.519716        2  Crítico
1                   Baja California       69.298142        0     Alto
24                          Sinaloa       63.857385        0     Alto
26                          Tabasco       63.768295        0     Alto
5                         Chihuahua       62.909854        0     Alto
27                       Tamaulipas       62.382713        0     Alto
12                          Hidalgo       56.925610        0     Alto
2               Baja California Sur       53.046589        0     Alto
16                           México       52.790884        0     Alto
6                  C

In [7]:
import io

# Save to S3
s3 = boto3.client('s3')
BUCKET = "mexico-public-safety-observatory"

csv_buffer = io.StringIO()
df_scores.to_csv(csv_buffer, index=False)

s3.put_object(
    Bucket=BUCKET,
    Key="outputs/violence_scores.csv",
    Body=csv_buffer.getvalue()
)

print("Saved to S3 successfully!")

Saved to S3 successfully!


In [9]:
# Score by state and year (for time series in Streamlit)
df_yearly = df.groupby(['entidad_nombre', 'anio', 'tipo_delito'])['rate_per_100k'].mean().reset_index()

df_yearly_wide = df_yearly.pivot_table(
    index=['entidad_nombre', 'anio'], 
    columns='tipo_delito', 
    values='rate_per_100k'
).fillna(0)

# Apply same scaler and PCA fitted on the full dataset
X_yearly_scaled = scaler.transform(df_yearly_wide)
X_yearly_pca = pca5.transform(X_yearly_scaled)

# Apply same weights
raw_score_yearly = X_yearly_pca @ weights

# Normalize using same min/max as before (important for consistency!)
score_yearly = (raw_score_yearly - score_min) / (score_max - score_min) * 100

# Build dataframe
df_time_series = pd.DataFrame({
    'entidad_nombre': df_yearly_wide.index.get_level_values('entidad_nombre'),
    'anio': df_yearly_wide.index.get_level_values('anio'),
    'violence_score': score_yearly
}).sort_values(['entidad_nombre', 'anio'])

#Clip
df_time_series['violence_score'] = df_time_series['violence_score'].clip(0, 100)

print(df_time_series.head(20))

     entidad_nombre  anio  violence_score
0    Aguascalientes  2015       16.405715
1    Aguascalientes  2016       16.098249
2    Aguascalientes  2017       25.622423
3    Aguascalientes  2018       43.774318
4    Aguascalientes  2019       46.248401
5    Aguascalientes  2020       49.627455
6    Aguascalientes  2021       45.218302
7    Aguascalientes  2022       47.359517
8    Aguascalientes  2023       34.955337
9    Aguascalientes  2024       37.360101
10   Aguascalientes  2025       27.654995
11  Baja California  2015       57.049934
12  Baja California  2016       48.770880
13  Baja California  2017       55.884385
14  Baja California  2018       63.719182
15  Baja California  2019       62.158911
16  Baja California  2020       67.115537
17  Baja California  2021       69.888857
18  Baja California  2022      100.000000
19  Baja California  2023      100.000000


In [10]:
csv_buffer2 = io.StringIO()
df_time_series.to_csv(csv_buffer2, index=False)

s3.put_object(
    Bucket=BUCKET,
    Key="outputs/violence_scores_yearly.csv",
    Body=csv_buffer2.getvalue()
)

print("Saved to S3 successfully!")

Saved to S3 successfully!
